In [13]:
import pandas as pd

df = pd.read_csv('../data/raw/agencies.csv')
df

,agency_id,agency_name,city,state,phone,founded_year,license_number
0,AGY-0001,Hall PLC Realty,Newark,New Jersey,+1-210-343-3218x1960,1989,RE-246316
1,AGY-0002,"Henderson, Ramirez and Lewis Realty",Virginia Beach,VA,2834863794,1980,RE-719176
2,AGY-0003,"Carter, Fuller and Mcclure Realty",Columbus,Ohio,001-851-316-1559x40781,2007,RE-731262
3,AGY-0004,Wood and Sons Realty,San Jose,CA,(993)710-3413x164,2009,RE-539898
4,AGY-0005,Trujillo Group Realty,Albany,NY,+1-241-592-8327x64835,1975,RE-895667
5,AGY-0006,"Richards, Hurst and Ross Realty",Spokane,WA,513-695-3767x242,1984,RE-325772
6,AGY-0007,James-Ferrell Realty,Tacoma,Washington,828.371.0122,1997,RE-988662
7,AGY-0008,"Gaines, Harrell and Evans Realty",Reading,PA,NaN,2004,RE-662275
8,AGY-0009,Koch-Decker Realty,San Antonio,Texas,+1-289-332-5288x0957,2015,RE-748564
9,AGY-0010,"Brennan, Henderson and Lewis Realty",Arvada,CO,430-639-1171x8227,1979,RE-148050


## Step 1: State Column Fix
The `state` column had inconsistent values — some were full names (e.g. `"New Jersey"`) and some were abbreviations (e.g. `"NJ"`). All values have been standardized to full state names.

In [15]:
# Map state abbreviations to full names
state_map = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
    'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho',
    'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
    'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
    'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
    'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
    'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia'
}

df['state'] = df['state'].map(lambda x: state_map.get(x, x))
df[['agency_id', 'city', 'state']].head(20)

,agency_id,city,state
0,AGY-0001,Newark,New Jersey
1,AGY-0002,Virginia Beach,Virginia
2,AGY-0003,Columbus,Ohio
3,AGY-0004,San Jose,California
4,AGY-0005,Albany,New York
5,AGY-0006,Spokane,Washington
6,AGY-0007,Tacoma,Washington
7,AGY-0008,Reading,Pennsylvania
8,AGY-0009,San Antonio,Texas
9,AGY-0010,Arvada,Colorado


## Step 2: Phone Column Fix
The `phone` column had 2 rows with missing values (NaN) — Row 7 and Row 38. Missing values have been replaced with `"Unknown"`.

In [16]:
# Fill missing phone values with "Unknown"
df['phone'] = df['phone'].fillna('Unknown')
df[['agency_id', 'agency_name', 'phone']].iloc[[7, 38]]

,agency_id,agency_name,phone
7,AGY-0008,"Gaines, Harrell and Evans Realty",Unknown
38,AGY-0039,Rodriguez-Johnson Realty,Unknown


## Step 3: Phone Format Standardize
The `phone` column had numbers in many different formats — plain digits, dashes, dots, parentheses, `+1` and `001` prefixes, and some with extensions (`x...`). All numbers have been standardized to `(XXX) XXX-XXXX` format. Extensions have been removed.

In [17]:
import re

def format_phone(phone):
    if phone == 'Unknown':
        return 'Unknown'

    # Strip extension (x...) and keep only the main number
    main = str(phone).lower().split('x')[0]

    # Remove all non-digit characters
    digits = re.sub(r'\D', '', main)

    # Remove country code: 001... or 1XXXXXXXXXX
    if digits.startswith('001'):
        digits = digits[3:]
    elif len(digits) == 11 and digits.startswith('1'):
        digits = digits[1:]

    # Format 10 digits as (XXX) XXX-XXXX
    if len(digits) == 10:
        return f'({digits[:3]}) {digits[3:6]}-{digits[6:]}'
    else:
        return phone  # return as-is if format is unrecognized

df['phone'] = df['phone'].apply(format_phone)
df[['agency_id', 'agency_name', 'phone']]

,agency_id,agency_name,phone
0,AGY-0001,Hall PLC Realty,(210) 343-3218
1,AGY-0002,"Henderson, Ramirez and Lewis Realty",(283) 486-3794
2,AGY-0003,"Carter, Fuller and Mcclure Realty",(851) 316-1559
3,AGY-0004,Wood and Sons Realty,(993) 710-3413
4,AGY-0005,Trujillo Group Realty,(241) 592-8327
5,AGY-0006,"Richards, Hurst and Ross Realty",(513) 695-3767
6,AGY-0007,James-Ferrell Realty,(828) 371-0122
7,AGY-0008,"Gaines, Harrell and Evans Realty",Unknown
8,AGY-0009,Koch-Decker Realty,(289) 332-5288
9,AGY-0010,"Brennan, Henderson and Lewis Realty",(430) 639-1171


In [18]:
# Check if any phone numbers still contain extensions (x...)
has_extension = df[df['phone'].str.contains('x', case=False, na=False)]
print(f"Rows with extensions: {len(has_extension)}")
has_extension[['agency_id', 'agency_name', 'phone']]

Rows with extensions: 0


,agency_id,agency_name,phone


## Step 4: Duplicates Check
Checking if `agency_id` and `license_number` columns have any duplicate values.

In [19]:
# Check for duplicate agency_id values
agency_dupes = df[df.duplicated(subset='agency_id', keep=False)]
print(f"Duplicate agency_id rows: {len(agency_dupes)}")
if len(agency_dupes) > 0:
    print(agency_dupes[['agency_id', 'agency_name']])

print()

# Check for duplicate license_number values
license_dupes = df[df.duplicated(subset='license_number', keep=False)]
print(f"Duplicate license_number rows: {len(license_dupes)}")
if len(license_dupes) > 0:
    print(license_dupes[['agency_id', 'agency_name', 'license_number']])

Duplicate agency_id rows: 0

Duplicate license_number rows: 0


## Step 5: Agency Name Fix
Two agency names had incorrect capitalization. Fixed manually.

In [20]:
# Fix incorrect capitalization in agency names
df.loc[df['agency_id'] == 'AGY-0043', 'agency_name'] = 'McGuire-Davis Realty'
df.loc[df['agency_id'] == 'AGY-0046', 'agency_name'] = "O'Neill, Henry and Salas Realty"

df.loc[df['agency_id'].isin(['AGY-0043', 'AGY-0046']), ['agency_id', 'agency_name']]

,agency_id,agency_name
42,AGY-0043,McGuire-Davis Realty
45,AGY-0046,"O'Neill, Henry and Salas Realty"


## Step 6: Save Cleaned Dataset
Saving the cleaned dataframe as `clean-agencies.csv` in the `data/cleaned/` folder.

In [21]:
import os

# Save cleaned dataset to data/cleaned folder
os.makedirs('../data/cleaned', exist_ok=True)
df.to_csv('../data/cleaned/clean-agencies.csv', index=False)
print("Saved: data/cleaned/clean-agencies.csv")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")

Saved: data/cleaned/clean-agencies.csv
Rows: 50, Columns: 7


In [22]:
df.head(3)

,agency_id,agency_name,city,state,phone,founded_year,license_number
0,AGY-0001,Hall PLC Realty,Newark,New Jersey,(210) 343-3218,1989,RE-246316
1,AGY-0002,"Henderson, Ramirez and Lewis Realty",Virginia Beach,Virginia,(283) 486-3794,1980,RE-719176
2,AGY-0003,"Carter, Fuller and Mcclure Realty",Columbus,Ohio,(851) 316-1559,2007,RE-731262


In [23]:

import pandas as pd
import re

df = pd.read_csv('../data/raw/agencies.csv')

# Step 1: State fix
state_map = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
    'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho',
    'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
    'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
    'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
    'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
    'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia'
}
df['state'] = df['state'].map(lambda x: state_map.get(x, x))

# Step 2: Phone missing
df['phone'] = df['phone'].fillna('Unknown')

# Step 3: Phone format
def format_phone(phone):
    if phone == 'Unknown':
        return 'Unknown'
    main = str(phone).lower().split('x')[0]
    digits = re.sub(r'\D', '', main)
    if digits.startswith('001'):
        digits = digits[3:]
    elif len(digits) == 11 and digits.startswith('1'):
        digits = digits[1:]
    if len(digits) == 10:
        return f'({digits[:3]}) {digits[3:6]}-{digits[6:]}'
    else:
        return phone

df['phone'] = df['phone'].apply(format_phone)

# Step 4: Agency name fix
df.loc[df['agency_id'] == 'AGY-0043', 'agency_name'] = 'McGuire-Davis Realty'
df.loc[df['agency_id'] == 'AGY-0046', 'agency_name'] = "O'Neill, Henry and Salas Realty"

# --- FINAL CHECKS ---
print("=== 1. Shape ===")
print(df.shape)

print("\n=== 2. Missing Values ===")
print(df.isnull().sum())

print("\n=== 3. Duplicate agency_id ===")
print(df.duplicated(subset='agency_id').sum())

print("\n=== 4. Duplicate license_number ===")
print(df.duplicated(subset='license_number').sum())

print("\n=== 5. State — unique values ===")
print(sorted(df['state'].unique()))

print("\n=== 6. Phone — any extensions left? ===")
print(df[df['phone'].str.contains('x', case=False, na=False)]['phone'].tolist())

print("\n=== 7. Phone — any non-standard format? ===")
import re
non_std = df[~df['phone'].str.match(r'^\(\d{3}\) \d{3}-\d{4}$') & (df['phone'] != 'Unknown')]
print(non_std[['agency_id', 'phone']])

print("\n=== 8. Agency names fixed? ===")
print(df.loc[df['agency_id'].isin(['AGY-0043', 'AGY-0046']), ['agency_id', 'agency_name']].to_string())

print("\n=== 9. founded_year range ===")
print(f"Min: {df['founded_year'].min()}, Max: {df['founded_year'].max()}, Nulls: {df['founded_year'].isnull().sum()}")


=== 1. Shape ===
(50, 7)

=== 2. Missing Values ===
agency_id         0
agency_name       0
city              0
state             0
phone             0
founded_year      0
license_number    0
dtype: int64

=== 3. Duplicate agency_id ===
0

=== 4. Duplicate license_number ===
0

=== 5. State — unique values ===
['Arizona', 'California', 'Colorado', 'Florida', 'Georgia', 'Michigan', 'New Jersey', 'New York', 'North Carolina', 'Ohio', 'Pennsylvania', 'Texas', 'Virginia', 'Washington']

=== 6. Phone — any extensions left? ===
[]

=== 7. Phone — any non-standard format? ===
Empty DataFrame
Columns: [agency_id, phone]
Index: []

=== 8. Agency names fixed? ===
   agency_id                      agency_name
42  AGY-0043             McGuire-Davis Realty
45  AGY-0046  O'Neill, Henry and Salas Realty

=== 9. founded_year range ===
Min: 1975, Max: 2019, Nulls: 0
